In [1]:
import time

import pandas as pd
from googleapiclient.discovery import build
from tqdm.auto import tqdm
from youtube_transcript_api import (
    YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled
)
from youtube_transcript_api.proxies import WebshareProxyConfig

from config.config import settings

YOUTUBE_API_KEY = settings.youtube_api_key
CHANNELS = settings.channels
MAX_VIDEOS_PER_CHANNEL = settings.max_videos_per_channel
LANGUAGE = settings.language
WEB_SHARE_USER = settings.web_share_username
WEB_SHARE_PW = settings.web_share_pw

In [2]:
youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

In [3]:
def resolve_channel_id(channel_input: str) -> dict:
    """
    Resolve a YouTube channel handle or ID to its channel ID and title.

    This function takes either a full channel ID (starting with 'UC') or a 
    channel handle (starting with '@') and returns a dictionary containing 
    the channel ID and the channel's title.

    Args:
        channel_input (str): The YouTube channel identifier, either a channel 
            ID (24 characters, starting with 'UC') or a handle (starting with '@').
    
    Returns:
        dict: A dictionary with keys:
            - 'channel_id' (str): The YouTube channel's unique ID.
            - 'channel_title' (str): The title of the channel.
    """
    if channel_input.startswith("UC") and len(channel_input) == 24:
        resp = youtube.channels().list(part="snippet", id=channel_input).execute()
    else:
        handle = channel_input.lstrip("@")
        resp = youtube.channels().list(part="snippet", forHandle=handle).execute()

    if not resp.get("items"):
        raise ValueError(f"Channel not found: {channel_input}")

    item = resp["items"][0]
    return {"channel_id": item["id"], "channel_title": item["snippet"]["title"]}

In [4]:
def get_uploads_playlist_id(channel_id: str) -> str:
    """
    Retrieve the uploads playlist ID for a given YouTube channel.

    Every YouTube channel has a special "uploads" playlist that contains all 
    videos uploaded by that channel. This function fetches the playlist ID 
    corresponding to that uploads playlist.

    Args:
        channel_id (str): The unique ID of the YouTube channel (starts with 'UC').

    Returns:
        str: The playlist ID of the channel's uploads playlist.
    """
    resp = youtube.channels().list(part="contentDetails", id=channel_id).execute()
    return resp["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

In [5]:
def get_videos_from_playlist(playlist_id: str, max_videos=None) -> list:
    """
    Fetch video details from a YouTube playlist.

    This function retrieves all videos from the specified playlist, returning 
    their video IDs, titles, and publication dates. It handles pagination 
    automatically and can optionally limit the number of videos returned.

    Args:
        playlist_id (str): The ID of the YouTube playlist to fetch videos from.
        max_videos (int, optional): The maximum number of videos to retrieve. 
            If None, all videos in the playlist are returned.
    
    Returns:
        list of dict: A list of dictionaries, each containing:
            - 'video_id' (str): The unique ID of the video.
            - 'title' (str): The title of the video.
            - 'published_at' (str): The ISO 8601 timestamp of when the video 
              was published.
    """
    videos = []
    next_page_token = None

    while True:
        resp = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token,
        ).execute()

        for item in resp["items"]:
            snippet = item["snippet"]
            videos.append({
                "video_id":     snippet["resourceId"]["videoId"],
                "title":        snippet["title"],
                "published_at": snippet["publishedAt"],
            })

        next_page_token = resp.get("nextPageToken")
        if not next_page_token:
            break
        if max_videos and len(videos) >= max_videos:
            break

    return videos[:max_videos] if max_videos else videos

In [6]:
def get_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetch the transcript text for a YouTube video.

    This function retrieves the transcript for the specified video in the 
    requested language. It first tries to fetch a manually created transcript, 
    and if unavailable, falls back to an automatically generated transcript. 
    If no transcript is available or an error occurs, it returns None.

    Args:
        video_id (str): The unique ID of the YouTube video.
        language (str, optional): Defaults to 'en'.
    
    Returns:
        str or None: The full transcript text as a single string, or None if 
        the transcript is unavailable.
    """
    try:
        ytt_api = YouTubeTranscriptApi(
        #      proxy_config=WebshareProxyConfig(
        #      proxy_username=WEB_SHARE_USER,
        #      proxy_password=WEB_SHARE_PW,
        #  )
        )
        transcript_list = ytt_api.list(video_id)
        
        # Try to find manual transcript first, then auto-generated
        try:
            transcript = transcript_list.find_transcript([language])
        except NoTranscriptFound:
            transcript = transcript_list.find_generated_transcript([language])
        
        segments = transcript.fetch()
        return " ".join(seg.text for seg in segments)

    except (NoTranscriptFound, TranscriptsDisabled):
        return None
    except Exception as e:
        print(f"  ⚠️  Error for {video_id}: {e}")
        return None

In [7]:
all_records = []

for channel_input in CHANNELS:
    print(f"\n📺 Processing: {channel_input}")

    try:
        channel_info  = resolve_channel_id(channel_input)
        channel_id    = channel_info["channel_id"]
        channel_title = channel_info["channel_title"]
        print(f"   Channel : {channel_title} ({channel_id})")

        playlist_id = get_uploads_playlist_id(channel_id)
        videos      = get_videos_from_playlist(playlist_id, max_videos=MAX_VIDEOS_PER_CHANNEL)
        print(f"   Videos  : {len(videos)} found")

    except Exception as e:
        print(f"   ❌ Could not fetch channel: {e}")
        continue

    for video in tqdm(videos, desc="  Transcripts"):
        transcript = get_transcript(video["video_id"], language=LANGUAGE)
        all_records.append({
            "channel":              channel_title,
            "channel_id":           channel_id,
            "video_id":             video["video_id"],
            "title":                video["title"],
            "published_at":         video["published_at"],
            "url":                  f"https://www.youtube.com/watch?v={video['video_id']}",
            "transcript":           transcript,
            "transcript_available": transcript is not None,
        })
        time.sleep(2)  # Be polite to the API

df = pd.DataFrame(all_records)
print(f"\n✅ Done! {len(df)} videos total.")
print(f"   Transcripts available: {df['transcript_available'].sum()} / {len(df)}")
df.head()


📺 Processing: @TwicebakedJake
   Channel : TwicebakedJake (UC8FyZ-7SuPVVaoxpFOE5pvA)
   Videos  : 1 found


  Transcripts:   0%|          | 0/1 [00:00<?, ?it/s]

  ⚠️  Error for fIJhjr5Lj_U: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=fIJhjr5Lj_U! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are us

  Transcripts:   0%|          | 0/1 [00:00<?, ?it/s]

  ⚠️  Error for u8CqdgoKBus: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=u8CqdgoKBus! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are us

,channel,channel_id,video_id,title,published_at,url,transcript,transcript_available
0,TwicebakedJake,UC8FyZ-7SuPVVaoxpFOE5pvA,fIJhjr5Lj_U,The BEST & WORST Value Pokemon Card Products R...,2026-02-24T21:00:45Z,https://www.youtube.com/watch?v=fIJhjr5Lj_U,None,False
1,okJLUV,UCNzKuyNGREMYkx-vBYTDkRg,u8CqdgoKBus,Card Artist Spotlight: Megumi Mizutani (Where ...,2026-02-21T11:34:16Z,https://www.youtube.com/watch?v=u8CqdgoKBus,None,False
